## Objetivo

Ensinar como escolher e comprovar a melhor estratégia de join em PySpark, reduzindo tempo e shuffle
- Ler o plano (explain) e entender BroadcastHashJoin vs SortMergeJoin
- Aplicar broadcast da dimensão, repartition/coalesce do fato e cache de reuso
- Lidar com skew (tendência de chave quente) e AQE (adaptive query execution)


## **Aula de Hoje — Joins, Skew e Shuffle** (Spark/Databricks)
>
Vamos alinhar conceitos e boas práticas para acelerar *joins*, reconhecer **skew** (chaves “quentes”) e reduzir **shuffle** (embaralhamento de dados). A meta é entender **o que acontece por baixo dos panos** e como **influenciar o plano** para ganhar tempo e estabilidade.
>
---
>
### 1) **Joins** — do lógico ao físico
>
**Tipos lógicos (o “que”)**: `INNER`, `LEFT/RIGHT`, `FULL`, `LEFT SEMI`, `LEFT ANTI`.
**Estratégias físicas (o “como”)**:
>
* **Broadcast Hash Join (BHJ)**: manda a tabela **pequena** para todos os executores → **evita shuffle** de um lado.
* **Sort Merge Join (SMJ)**: **embaralha e ordena** pelas chaves nos dois lados; bom quando **ambos** os lados são grandes.
* **Shuffle Hash Join (SHJ)**: particiona por chave e usa hash; também requer **shuffle**.
>
**Como influenciar (sem mexer em configs)**:
PySpark → `df.join(broadcast(dim), "key")` (força BHJ)
Hints para evitar broadcast → `.hint("merge")` (tende a SMJ) ou `.hint("shuffle_hash")`
>
**Armadilhas**: chaves com tipos diferentes, nulos não tratados, joins redundantes (multiplicação de linhas).
>
---
>
### 2) **Skew** — desequilíbrio de chaves
>
**O que é**: poucas chaves concentram **muitas linhas** → partições pesadas, tarefas lentas.
**Como identificar**:
>
* Exploratória: `groupBy(key).count().orderBy(desc("count"))` (picos evidentes = skew).
* Spark UI: tarefas com tempo muito maior em poucas partições; *spill* e *shuffle* desbalanceados.
>
**Mitigações**:
>
* **Broadcast** do lado pequeno.
* **Salting** (subchaves) para chaves quentes.
* **Pré-agg / filtros** antes do join.
* **repartition(key)** para balancear (a custo de um shuffle controlado).
>
**Sinal de sucesso**: tempo total cai e tarefas ficam mais homogêneas no Spark UI.
>
---
>
### 3) **Shuffle** — o custo oculto
>
**O que é**: redistribuição de dados pela rede/disco para agrupar/ordenar por chaves.

**Quando ocorre**: joins **não-broadcast**, `groupBy/agg`, `distinct`, `orderBy/sort`, `repartition`.

**Por que dói**: I/O de disco e rede, arquivos temporários, possíveis *spills*.
>
**Como reduzir**:
>
* **Broadcast** quando possível.
* **Filtrar/projetar cedo** (`filter`/`select`) antes de join/agg.
* **Repartition** alinhado à chave antes de operações grandes.
* **Coalesce** após grande redução (compacta sem novo shuffle).
* **Cache** para reuso de resultados (evita refazer o mesmo shuffle).
>


# Lazy Evaluation no Spark

**Ideia central:** o Spark **não executa** suas operações imediatamente. Ele constrói um **plano lógico** a partir de *transformations* e **só executa** quando você chama uma *action*.
Vantagens: melhor **otimização**, menos I/O desnecessário e **controle** de quando pagar o custo da computação.

---

### 1) O que conta como *Transformation* (não executa agora)

Gera um **novo DataFrame** apenas adicionando passos ao plano.

* `select(...)`, `withColumn(...)`, `drop(...)`
* `filter(...)` / `where(...)`
* `join(...)`
* `orderBy(...)`
* `repartition(...)` / `coalesce(...)`
* `groupBy(...).agg(...)` *(define o plano, mas continua preguiçoso até uma action)*

**Exemplo:**

~~~python
from pyspark.sql import functions as F

df_trf = (
  df_raw
    .filter(F.col("status") == "OK")
    .withColumn("price_net", F.col("price") * (1 - F.col("discount")))
    .select("id", "price_net", "ts")
)
# Até aqui, nada foi executado. Só temos o plano.
~~~

---

### 2) O que conta como *Action* (dispara execução)

Materializa resultados, dispara **jobs/stages** e lê/processa dados.

* **Leituras:** `show()`, `head()`, `take(n)`, `collect()`, `count()`, `first()`
* **Escritas:** `write.format("delta").save(...)`, `saveAsTable(...)`, `insertInto(...)`
* **Databricks:** `display(df)` também **executa** o plano

**Exemplo (dispara execução):**

```python
df_trf.show(10)     # executa
df_trf.count()      # executa
display(df_trf)     # executa
```
---

### 3) Por que a *Lazy Evaluation* importa

* **Otimização:** o Catalyst/AQE pode reordenar filtros, eliminar colunas não usadas, escolher estratégia de join.
* **Performance & custo:** você evita leituras e shuffles desnecessários até o momento certo.
* **Depuração:** `explain("formatted")` mostra o plano antes de gastar recursos (vamos usar bastante, hoje).

---

### 4) Perguntas rápidas (FAQ)

* **`explain` é action?** Não. Ele imprime o **plano**; não materializa todos os dados.
* **`cache()` executa?** Não. Precisa de uma *action* (ex.: `count()`) para materializar no cache.
* **`display(df)` executa?** Sim, `display` é uma *action* no Databricks.
* **`repartition` executa?** É *transformation*; o shuffle só acontece quando uma *action* dispara o job.

---

> **Resumo:** construa o pipeline com *transformations*, valide o plano com `explain`, e só então chame *actions* quando realmente precisar dos resultados.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
import time

SCALE_FACT = 3_000_000 # quantidade de linhas que teremos na fato
N_PRODUCTS = 3_000 # quantidade de produtos
N_CUSTOMERS = 100_000 # quantidade de clientes 
SEED = 42 # seed para aleatoriedade e reproducibilidade

def bench(df, name="job"):
    t0 = time.perf_counter()
    out = df.count()              # força execução (action)
    dt = time.perf_counter() - t0
    print(f"[{name}] count={out} | {dt:.2f}s")
    return dt


In [0]:
# Dimensão: produtos
dim_products = (
    spark.range(1, N_PRODUCTS + 1).toDF("product_id")
    .withColumn("category_id", (F.rand(SEED)*10).cast("int"))
    .withColumn("brand_id", (F.rand(SEED+1)*50).cast("int"))
    .withColumn("price_list", (F.round(F.rand(SEED+2)*900 + 100, 2)).cast("double"))
    .withColumn("product_name", F.concat(F.lit("prod_"), F.col("product_id")))
)


In [0]:
# Dimensão: clientes (opcional)
dim_customers = (
    spark.range(1, N_CUSTOMERS + 1).toDF("customer_id")
    .withColumn("country", F.when(F.rand(SEED) < 0.6, F.lit("BR")) # F.lit(random.choice([...]))
                            .when(F.rand(SEED) < 0.8, F.lit("MX"))
                            .otherwise(F.lit("AR")))
)


In [0]:

# Fato: vendas (skew proposital em product_id=1 ~20%)
fact_sales = (
    spark.range(1, SCALE_FACT + 1).toDF("sale_id")
    .withColumn("product_id",
        F.when(F.rand(SEED) < 0.20, F.lit(1))  # 20% na mesma chave
         .otherwise((F.rand(SEED+3) * (N_PRODUCTS-1) + 2).cast("int"))
    )
    .withColumn("customer_id", (F.rand(SEED+4) * N_CUSTOMERS + 1).cast("int"))
    .withColumn("ts", F.expr("timestampadd(MICROSECOND, cast(rand()*1e12 as bigint), timestamp'2025-01-01 00:00:00')"))
    .withColumn("qty", (F.rand(SEED+5)*9 + 1).cast("int"))  # 1 a 10
    .withColumn("discount", F.round(F.rand(SEED+6)*0.25, 2))
)


In [0]:
print(f"dim_products={dim_products.count()}, dim_customers={dim_customers.count()}, fact_sales={fact_sales.count()}")

In [0]:
dim_products.createOrReplaceTempView("dim_products")
dim_customers.createOrReplaceTempView("dim_customers")
fact_sales.createOrReplaceTempView("fact_sales")


In [0]:
%sql
SELECT *
FROM dim_customers

In [0]:
%sql
SELECT
  product_name,
  country,
  SUM(p.price_list * qty) AS TOTAL_TICKET,
  AVG(price_list) AS average_sales_price,
  AVG(qty) AS average_quantity
FROM fact_sales s
LEFT JOIN dim_products p
  ON p.product_id = s.product_id
LEFT JOIN dim_customers c
  ON c.customer_id = s.customer_id
WHERE p.product_id = 1001
GROUP BY ALL
ORDER BY product_name, country


Databricks visualization. Run in Databricks to view.

In [0]:
no_broadcast = (fact_sales.hint("merge")
                .join(dim_products.hint("merge"), "product_id", "inner"))
no_broadcast.explain("formatted")
t_nb = bench(no_broadcast, "no_broadcast_hint")  # tende a virar SortMergeJoin


In [0]:
from pyspark.sql.functions import broadcast

with_broadcast = fact_sales.join(broadcast(dim_products), "product_id", "inner") # simplesmente adicionamos a função "broadcast" no nosso join
with_broadcast.explain("formatted")
t_b = bench(with_broadcast, "with_broadcast")   # deve mostrar BroadcastHashJoin


In [0]:
from pyspark.sql import functions as F

# Tente 64 KB por linha (~64 * 3000 ≈ 192 MB). Se ainda vier BHJ, suba para 128 ou 256.
KB_PER_ROW = 64
dim_products_heavy = dim_products.withColumn(
    "payload",
    F.expr(f"repeat('x', {KB_PER_ROW} * 1024)")
)

# Dê hint para preferir SortMergeJoin
no_broadcast_heavy = (fact_sales
                      .hint("merge")
                      .join(dim_products_heavy.hint("merge"), "product_id", "inner"))

no_broadcast_heavy.explain("formatted")
t_nb_heavy = bench(no_broadcast_heavy, "no_broadcast_heavy")


In [0]:

fact_repart = fact_sales.repartition(200, "product_id")  # ajuste 200 ao tamanho do cluster
repart_broadcast = fact_repart.join(broadcast(dim_products), "product_id")
repart_broadcast.explain("formatted")
t_rep = bench(repart_broadcast, "repartition+broadcast")

dim_small = (
    dim_products
    .select("product_id", "category_id", "brand_id", "price_list")
    # .cache() # fora do servelss, é possível cacher para aumentar a performance de consultas recorrentes
)
# dim_small.count()  # você SEMPRE precisa materializar o seu persist

cached_q = (fact_sales.join(broadcast(dim_small), "product_id")
            .groupBy("category_id").agg(F.sum("qty").alias("qty_total")))
t_c1 = bench(cached_q, "cached_1")
t_c2 = bench(cached_q, "cached_2")  # a repetição gera ganho, mesmo no serverless, pelas otimizações do Databricks

# dim_small.unpersist()

# c) Coalesce após reduzir muito o volume (sem shuffle)
reduced = with_broadcast.filter(F.col("brand_id") == 1)
t_red = bench(reduced, "reduced_filter")
reduced_coalesced = reduced.coalesce(16)
reduced_coalesced.count()


In [0]:
# como retornar o número de partições de um Dataframe? 
try:
    print(dim_small.rdd.getNumPartitions())
    print(cached_q.rdd.getNumPartitions())
    print(reduced.rdd.getNumPartitions())
    print(with_broadcast.rdd.getNumPartitions())
    print(reduced_coalesced.rdd.getNumPartitions())
except Exception as e:
    print(e)

In [0]:
from pyspark.sql import functions as F

N_SALT = 8

dim_salted = (
    dim_products
    .withColumn(
        "salt_candidates",
        F.when(F.col("product_id") == 1,
               F.array(*[F.lit(i) for i in range(N_SALT)]))   # cria um array condicional
         .otherwise(F.array(F.lit(0)))
    )
    .withColumn("salt", F.explode("salt_candidates"))          # explode no topo
    .drop("salt_candidates")
)

fact_salted = (
    fact_sales
    .withColumn(
        "salt",
        F.when(F.col("product_id") == 1, (F.rand(SEED) * N_SALT).cast("int"))
         .otherwise(F.lit(0))
    )
)

# 3) Join salgado + broadcast da dimensão
from pyspark.sql.functions import broadcast

salted_join = fact_salted.join(broadcast(dim_salted), ["product_id", "salt"], "inner")
bench(salted_join, "salted+broadcast")


In [0]:
display(
    salted_join
    .groupBy('salt')
    .count()
)

Databricks visualization. Run in Databricks to view.